In [ ]:
import numpy as np
import scipy.stats as stats
import xarray as xr
from xclim.core.units import convert_units_to
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import pandas as pd
import seaborn as sns
import climakitae as ck
from typing import Tuple, Dict, Union, Optional
import geopandas as gpd
import intake

**Quantile Delta Mapping (QDM)** is a bias-correction technique designed to adjust model outputs so their statistical distributions match real-world observations, all while preserving the raw model's projected long-term trend.

---

### Step 1: Reshape into a 3D Calendar Grid

Before computing statistical distributions, the 1D continuous time series (spanning many years) is unstacked into a **3D grid** organized by:

* **Year** (e.g., 1980, 1981, …)
* **Day of Year** ($1 \dots 365$)
* **Hour of Day** ($0 \dots 23$)

This structure isolates seasonal and diurnal patterns across the record.

---

### Step 2: Apply Circular Boundary Padding

Weather patterns transition smoothly across time boundaries. To prevent artificial gaps at calendar edges:

* **Day-of-Year Padding:** The end of December is wrapped around to pad early January, and vice versa.
* **Hour-of-Day Padding:** Late-night hours ($23:00$) are wrapped around to pad early morning hours ($00:00$).

---

### Step 3: Pool Data into Rolling Windows

Rather than evaluating each hour or day in strict isolation (which might not contain enough historical data points for stable statistics), data is gathered using a moving window pool:

* **Day-of-Year Windowing:** A multi-day window (e.g., 91 days) centered on a target day of the year pools neighboring days across all available historical years.
* **Hour-of-Year Windowing:** A multi-hour window (e.g., 5 hours) centered on a target hour of day pools neighboring hours around the target time (e.g., $11:00 \text{ to } 15:00$) alongside the multi-day window, capturing diurnal variability with increased sample sizes.

All values falling within the configured rolling windows across all reference years are combined into a single statistical sample pool for that specific calendar point.

---

### Step 4: Compute Quantile Matrices

For every pooled slice $(d, h)$, the algorithm calculates sample quantiles across a probability grid $q \in [0.01, 0.99]$ for three datasets:

1. **$Q_{\text{obs}}$:** Observed baseline values (the ground truth target).
2. **$Q_{\text{ref}}$:** Raw historical model values (the model's baseline performance).
3. **$Q_{\text{proj}}$:** Raw future/projection model values (the raw model output to be corrected).

---

### Step 5: Quantile Delta Mapping (The Correction Core)

For every individual value in the target model time series $x_{\text{model}}$ at day $d$ and hour $h$:

1. **Find Model Rank:** Determine the percentile rank $q$ of $x_{\text{model}}$ within its historical model reference distribution $Q_{\text{ref}}$.
2. **Map to Observations:** Find the observed value $x_{\text{obs}}$ corresponding to that same percentile rank $q$ in $Q_{\text{obs}}$.
3. **Calculate Model Trend (Delta):** Compute the relative change predicted by the model for that percentile:
* **Additive Mode:** $\Delta = x_{\text{model}} - Q_{\text{ref}}(q)$
* **Multiplicative Mode:** $\Delta = \frac{x_{\text{model}}}{Q_{\text{ref}}(q)}$


4. **Combine Mapped Value & Delta:** Apply the model's projected delta to the observed value:
* **Additive Final:** $x_{\text{adjusted}} = x_{\text{obs}} + \Delta$
* **Multiplicative Final:** $x_{\text{adjusted}} = x_{\text{obs}} \times \Delta$



---

### Step 6: Flatten Back to a 1D Time Series

Finally, the adjusted 3D grid $(\text{year}, \text{dayofyear}, \text{hour})$ is stacked back down into a continuous 1D time series matching the original datetime index and metadata attributes.

In [ ]:
# Define visual style palette
COLOR_OBS = "#000000"       # Black (Observed Baseline)
COLOR_RAW_REF = "#D55E00"   # Vermillion (Raw Historical Model Reference)
COLOR_QDM_REF = "#E69F00"   # Orange (QDM Bias-Adjusted Reference)
COLOR_RAW_PROJ = "#0072B2"  # Deep Blue (Raw Future Model Projection)
COLOR_QDM_PROJ = "#009E73"  # Bluish Green (QDM Bias-Adjusted Projection)


######### Support functions ##########
# =====================================================================
# BASIC QA FUNCTIONS
# =====================================================================

def apply_qaqc(
    obs_ds: xr.Dataset,
    stn_name: str,
    qaqc_suffix: str = "_eraqc",
) -> xr.Dataset:
    """Apply quality control flags to mask invalid observations.

    Parameters
    ----------
    obs_ds : xr.Dataset
        Raw input dataset containing meteorological variables and QAQC flags.
    stn_name : str
        Station identifier name for figure titling.
    qaqc_suffix : str, default="_eraqc"
        Suffix identifying quality control flag variables.

    Returns
    -------
    xr.Dataset
        Cleaned Dataset with flagged observations set to NaN.
    """
    clean_ds = xr.Dataset(attrs=obs_ds.attrs)
    qa_vars = [var for var in obs_ds.data_vars if var.endswith(qaqc_suffix)]
    base_vars = [
        var.replace(qaqc_suffix, "")
        for var in qa_vars
        if var.replace(qaqc_suffix, "") in obs_ds
    ]

    # Mask base variable data where QAQC flag condition is populated
    for var in base_vars:
        qa_var = f"{var}{qaqc_suffix}"
        condition = obs_ds[qa_var].notnull()
        clean_ds[var] = obs_ds[var].where(~condition, np.nan)

    return clean_ds

def _pad_dayofyear_cyclically(
    da_3d: xr.DataArray, window_size: int
) -> xr.DataArray:
    """Pad a 3D (year, dayofyear, hour) DataArray circularly along dayofyear.

    Parameters
    ----------
    da_3d : xarray.DataArray
        Input 3D DataArray with dimensions ('year', 'dayofyear', 'hour').
    window_size : int
        Total width of the rolling day-of-year window.

    Returns
    -------
    xarray.DataArray
        DataArray with padded day-of-year boundary values attached to both ends.
    """
    # 1. Calculate how many days to wrap around calendar boundaries
    half_win = window_size // 2

    # 2. Extract late December values to attach before Jan 1 (head padding)
    head_pad = da_3d.isel(dayofyear=slice(-half_win, None)).copy()

    # 3. Extract early January values to attach after Dec 31 (tail padding)
    tail_pad = da_3d.isel(dayofyear=slice(0, half_win)).copy()

    # 4. Relabel coordinate values so xarray concatenates cleanly
    min_doy = da_3d.dayofyear.values[0]
    max_doy = da_3d.dayofyear.values[-1]

    head_pad["dayofyear"] = np.arange(min_doy - half_win, min_doy)
    tail_pad["dayofyear"] = np.arange(max_doy + 1, max_doy + 1 + half_win)

    # 5. Concatenate head padding + original data + tail padding
    return xr.concat([head_pad, da_3d, tail_pad], dim="dayofyear")


def _pad_hour_cyclically(
    da_3d: xr.DataArray, window_size: int
) -> xr.DataArray:
    """Pad a 3D (year, dayofyear, hour) DataArray circularly along hour.

    Parameters
    ----------
    da_3d : xarray.DataArray
        Input 3D DataArray with dimensions ('year', 'dayofyear', 'hour').
    window_size : int
        Total width of the rolling hour window.

    Returns
    -------
    xarray.DataArray
        DataArray with padded hour boundary values attached to both ends.
    """
    # 1. Calculate how many hours to wrap around midnight
    half_win = window_size // 2

    # 2. Wrap late-night hours around to before midnight (head padding)
    head_pad = da_3d.isel(hour=slice(-half_win, None)).copy()

    # 3. Wrap early-morning hours around to after 23:00 (tail padding)
    tail_pad = da_3d.isel(hour=slice(0, half_win)).copy()

    # 4. Relabel hour coordinate values to preserve proper sequence
    min_hour = da_3d.hour.values[0]
    max_hour = da_3d.hour.values[-1]

    head_pad["hour"] = np.arange(min_hour - half_win, min_hour)
    tail_pad["hour"] = np.arange(max_hour + 1, max_hour + 1 + half_win)

    # 5. Concatenate head padding + original data + tail padding
    return xr.concat([head_pad, da_3d, tail_pad], dim="hour")


def _to_year_doy_hour_grid(
    ds: Union[xr.DataArray, xr.Dataset]
) -> xr.DataArray:
    """Convert a 1D time-series DataArray into a 3D (year, doy, hour) array.

    Parameters
    ----------
    ds : xarray.DataArray or xarray.Dataset
        Input 1D time-series data indexed by time.

    Returns
    -------
    xarray.DataArray
        Reshaped 3D DataArray.
    """
    # 1. Extract calendar year, day of year, and hour of day
    year = ds.time.dt.year
    doy = ds.time.dt.dayofyear
    hour = ds.time.dt.hour

    # 2. Construct a Pandas MultiIndex combining temporal components
    index = pd.MultiIndex.from_arrays(
        [year.values, doy.values, hour.values],
        names=["year", "dayofyear", "hour"],
    )

    # 3. Attach MultiIndex and unstack into a clean 3D grid
    da_indexed = xr.DataArray(
        ds.values, coords={"index": index}, dims=["index"]
    ).unstack("index")

    return da_indexed


def _unstack_to_da(
    corr_3d: xr.DataArray, ds: Union[xr.DataArray, xr.Dataset]
) -> xr.DataArray:
    """Flatten a 3D (year, dayofyear, hour) DataArray back into a 1D series.

    Parameters
    ----------
    corr_3d : xarray.DataArray
        Adjusted 3D DataArray.
    ds : xarray.DataArray or xarray.Dataset
        Reference 1D input supplying target time coordinates and metadata.

    Returns
    -------
    xarray.DataArray
        1D DataArray indexed by the original time coordinate.
    """
    # 1. Stack the 3D grid back down into a single continuous time dimension
    flattened = corr_3d.stack(
        time=("year", "dayofyear", "hour")
    ).dropna("time")

    # 2. Re-assign original 1D time coordinates and metadata attributes
    return xr.DataArray(
        flattened.values,
        coords={"time": ds.time.values},
        dims=["time"],
        attrs=ds.attrs,
    )


def _apply_qdm(
    target_3d: xr.DataArray,
    q_ref: xr.DataArray,
    q_obs: xr.DataArray,
    q_grid: np.ndarray,
    kind: str = "additive",
) -> xr.DataArray:
    """Apply Quantile Delta Mapping adjustment factors to a 3D dataset.

    Parameters
    ----------
    target_3d : xarray.DataArray
        Input 3D model dataset (year, dayofyear, hour) to bias-adjust.
    q_ref : xarray.DataArray
        Reference model quantile matrix.
    q_obs : xarray.DataArray
        Observed baseline quantile matrix.
    q_grid : numpy.ndarray
        1D array of quantile probability levels (0..1).
    kind : {'additive', 'multiplicative'}, default 'additive'
        Type of bias correction mode to apply.

    Returns
    -------
    xarray.DataArray
        Adjusted 3D DataArray with bias corrections applied.
    """
    adjusted_3d = xr.full_like(target_3d, np.nan)
    has_hour_dim = "hour" in q_ref.dims

    for doy in range(1, 366):
        for h in range(24):
            t_vals = target_3d.sel(dayofyear=doy, hour=h).values

            if np.all(np.isnan(t_vals)):
                continue

            if has_hour_dim:
                q_ref_vec = q_ref.sel(dayofyear=doy, hour=h).values
                q_obs_vec = q_obs.sel(dayofyear=doy, hour=h).values
            else:
                q_ref_vec = q_ref.sel(dayofyear=doy).values
                q_obs_vec = q_obs.sel(dayofyear=doy).values

            # Step A: Estimate percentile rank
            target_ranks = np.interp(t_vals, q_ref_vec, q_grid)

            # Step B: Map rank to observed baseline
            obs_mapped = np.interp(target_ranks, q_grid, q_obs_vec)

            # Step C: Compute model delta and apply correction
            if kind == "additive":
                target_delta = t_vals - np.interp(
                    target_ranks, q_grid, q_ref_vec
                )
                adjusted_3d.loc[dict(dayofyear=doy, hour=h)] = (
                    obs_mapped + target_delta
                )
            else:
                ref_mapped = np.interp(target_ranks, q_grid, q_ref_vec)
                target_ratio = t_vals / np.where(
                    ref_mapped == 0, 1e-6, ref_mapped
                )
                adjusted_3d.loc[dict(dayofyear=doy, hour=h)] = (
                    obs_mapped * target_ratio
                )

    return adjusted_3d


######## QDM pipeline #########


def run_qdm_pipeline(
    ds_obs: Union[xr.DataArray, xr.Dataset],
    ds_ref: Union[xr.DataArray, xr.Dataset],
    ds_proj: Optional[Union[xr.DataArray, xr.Dataset]] = None,
    day_window: int = 91,
    hour_window: Optional[int] = 5,
    n_quantiles: int = 99,
    kind: str = "additive",
    return_intermediate: bool = True,
) -> Union[
    Tuple[xr.DataArray, xr.DataArray],
    Tuple[xr.DataArray, xr.DataArray, Dict[str, xr.DataArray]],
]:
    """Execute Quantile Delta Mapping (QDM) with circular rolling windows.

    Always computes and returns bias-adjusted versions for both the baseline
    reference (`ds_ref`) and optional future projection (`ds_proj`).

    Parameters
    ----------
    ds_obs : xarray.DataArray or xarray.Dataset
        Observed historical baseline time series.
    ds_ref : xarray.DataArray or xarray.Dataset
        Model historical baseline reference time series.
    ds_proj : xarray.DataArray or xarray.Dataset, optional
        Model future projection time series. Default is copy of `ds_ref`.
    day_window : int, default 91
        Width of rolling day-of-year window for pooling quantiles.
    hour_window : int, optional, default 5
        Width of rolling hour window for pooling quantiles.
    n_quantiles : int, default 99
        Number of quantile probability levels between 0.01 and 0.99.
    kind : {'additive', 'multiplicative'}, default 'additive'
        Type of bias correction mode to apply.
    return_intermediate : bool, default True
        Whether to return intermediate quantile matrices and diagnostics.

    Returns
    -------
    ds_qdm_proj : xarray.DataArray
        QDM bias-adjusted future projection time series.
    ds_qdm_ref : xarray.DataArray
        QDM bias-adjusted historical baseline reference time series.
    diagnostics : dict of {str: xarray.DataArray}, optional
        Diagnostic outputs (e.g. quantile matrices).
    """
    if ds_proj is None:
        ds_proj = ds_ref.copy()

    q_grid = np.linspace(0.01, 0.99, n_quantiles)
    half_win_day = day_window // 2

    obs_3d = _to_year_doy_hour_grid(ds_obs)
    ref_3d = _to_year_doy_hour_grid(ds_ref)
    proj_3d = _to_year_doy_hour_grid(ds_proj)

    obs_padded = _pad_dayofyear_cyclically(obs_3d, day_window)
    ref_padded = _pad_dayofyear_cyclically(ref_3d, day_window)
    proj_padded = _pad_dayofyear_cyclically(proj_3d, day_window)

    if hour_window is not None and hour_window > 1:
        half_win_hour = hour_window // 2
        obs_padded = _pad_hour_cyclically(obs_padded, hour_window)
        ref_padded = _pad_hour_cyclically(ref_padded, hour_window)
        proj_padded = _pad_hour_cyclically(proj_padded, hour_window)

    obs_win = obs_padded.rolling(
        dayofyear=day_window, center=True
    ).construct("win_day")
    ref_win = ref_padded.rolling(
        dayofyear=day_window, center=True
    ).construct("win_day")
    proj_win = proj_padded.rolling(
        dayofyear=day_window, center=True
    ).construct("win_day")

    obs_win = obs_win.isel(
        dayofyear=slice(half_win_day, half_win_day + 365)
    )
    ref_win = ref_win.isel(
        dayofyear=slice(half_win_day, half_win_day + 365)
    )
    proj_win = proj_win.isel(
        dayofyear=slice(half_win_day, half_win_day + 365)
    )

    if hour_window is not None and hour_window > 1:
        obs_win = obs_win.rolling(
            hour=hour_window, center=True
        ).construct("win_hour")
        ref_win = ref_win.rolling(
            hour=hour_window, center=True
        ).construct("win_hour")
        proj_win = proj_win.rolling(
            hour=hour_window, center=True
        ).construct("win_hour")

        obs_win = obs_win.isel(
            hour=slice(half_win_hour, half_win_hour + 24)
        )
        ref_win = ref_win.isel(
            hour=slice(half_win_hour, half_win_hour + 24)
        )
        proj_win = proj_win.isel(
            hour=slice(half_win_hour, half_win_hour + 24)
        )
        pool_dims = ["year", "win_day", "win_hour"]
    else:
        pool_dims = ["year", "win_day", "hour"]

    q_obs = obs_win.quantile(
        q_grid, dim=pool_dims, 
    ).rename({"quantile": "q"})
    q_ref = ref_win.quantile(
        q_grid, dim=pool_dims, 
    ).rename({"quantile": "q"})
    q_proj = proj_win.quantile(
        q_grid, dim=pool_dims,
    ).rename({"quantile": "q"})

    if kind == "additive":
        delta_matrix = q_obs - q_ref
    else:
        delta_matrix = q_obs / np.where(q_ref == 0, 1e-6, q_ref)

    adjusted_ref_3d = _apply_qdm(ref_3d, q_ref, q_obs, q_grid, kind=kind)
    adjusted_proj_3d = _apply_qdm(proj_3d, q_ref, q_obs, q_grid, kind=kind)

    ds_qdm_ref = _unstack_to_da(adjusted_ref_3d, ds_ref)
    ds_qdm_proj = _unstack_to_da(adjusted_proj_3d, ds_proj)

    diagnostics = {
        "q_grid": q_grid,
        "q_obs": q_obs,
        "q_ref": q_ref,
        "q_proj": q_proj,
        "delta_matrix": delta_matrix,
        "correction_mode": kind,
    }

    if return_intermediate:
        return ds_qdm_proj, ds_qdm_ref, diagnostics
    return ds_qdm_proj, ds_qdm_ref


######## Diagnostic Plotting Functions #########


def _compute_hourly_mean_heatmap(
    ds: Union[xr.DataArray, xr.Dataset]
) -> np.ndarray:
    """Compute a (24, 365) hour x dayofyear mean climatology matrix directly 
    from time-series data.

    Calculates the mean value across years for each hour of each day of the year,
    filtering out leap days (Feb 29) to produce a smooth (24, 365) climatology.

    Parameters
    ----------
    ds : xarray.DataArray or xarray.Dataset
        Input 1D time-series data indexed by time.

    Returns
    -------
    numpy.ndarray
        2D mean climatology matrix of shape (24, 365).
    """
    da = ds.to_array().squeeze() if isinstance(ds, xr.Dataset) else ds
    clean = da.sel(time=~((da.time.dt.month == 2) & (da.time.dt.day == 29)))

    year = clean.time.dt.year
    doy = clean.time.dt.dayofyear
    hour = clean.time.dt.hour

    index = pd.MultiIndex.from_arrays(
        [year.values, doy.values, hour.values],
        names=["year", "dayofyear", "hour"],
    )

    da_3d = xr.DataArray(
        clean.values, coords={"index": index}, dims=["index"]
    ).unstack("index")

    # Take mean across all years for each (dayofyear, hour) coordinate
    mean_2d = da_3d.mean(dim="year")
    matrix = mean_2d.transpose("hour", "dayofyear").values

    return matrix[:, :365]


def _extract_quantile_heatmap(
    diagnostics: Dict[str, xr.DataArray],
    key: str,
    eval_quantile: float = 0.50,
) -> np.ndarray:
    """Extract a (24, 365) matrix directly from QDM pipeline diagnostics.

    Parameters
    ----------
    diagnostics : dict of {str: xarray.DataArray}
        Pipeline diagnostics dictionary returned by `run_qdm_pipeline`.
    key : str
        Target diagnostic dictionary key (e.g., 'q_obs', 'q_ref', 'q_proj').
    eval_quantile : float, default 0.50
        Quantile level (0.0 to 1.0) to evaluate.

    Returns
    -------
    numpy.ndarray
        2D matrix of shape (24, 365).
    """
    q_da = diagnostics[key]
    q_dim = "q" if "q" in q_da.dims else "quantile"
    q_slice = q_da.sel({q_dim: eval_quantile}, method="nearest")

    day_dim = (
        "dayofyear"
        if "dayofyear" in q_slice.dims
        else ("day" if "day" in q_slice.dims else None)
    )
    hour_dim = "hour" if "hour" in q_slice.dims else None

    if day_dim and hour_dim:
        matrix = q_slice.transpose(hour_dim, day_dim).values
        return matrix[:, :365]

    if day_dim and not hour_dim:
        daily_vec = q_slice.values[:365]
        return np.tile(daily_vec, (24, 1))

    return np.zeros((24, 365))


def _roll_to_pst(matrix: np.ndarray, pst_offset: int = -8) -> np.ndarray:
    """Roll a 2D (24, 365) matrix along the hour axis to convert to PST."""
    return np.roll(matrix, pst_offset, axis=0)


def _get_monthly_diurnal_cycle_pst(
    da: xr.DataArray, gap_width: int = 2, pst_offset: int = -8
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, list]:
    """Compute the monthly mean diurnal profile in PST with NaN gaps."""
    clean = da.sel(time=~((da.time.dt.month == 2) & (da.time.dt.day == 29)))
    monthly_diurnal = clean.groupby("time.month").map(
        lambda m_da: m_da.groupby("time.hour").mean("time")
    )
    diurnal_pst = np.roll(monthly_diurnal.values, pst_offset, axis=1)

    series_with_gaps = []
    x_coords = []
    month_centers = []
    current_x = 0

    month_labels = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]

    for m in range(12):
        month_vals = diurnal_pst[m, :]
        series_with_gaps.extend(month_vals)
        x_coords.extend(range(current_x, current_x + 24))

        month_centers.append(current_x + 11.5)
        current_x += 24

        if m < 11:
            series_with_gaps.extend([np.nan] * gap_width)
            x_coords.extend(range(current_x, current_x + gap_width))
            current_x += gap_width

    return (
        np.array(x_coords),
        np.array(series_with_gaps),
        np.array(month_centers),
        month_labels,
    )


def plot_qdm_diagnostics(
    ds_obs: Union[xr.DataArray, xr.Dataset],
    ds_ref: Union[xr.DataArray, xr.Dataset],
    ds_qdm_ref: Union[xr.DataArray, xr.Dataset],
    ds_proj: Optional[Union[xr.DataArray, xr.Dataset]] = None,
    ds_qdm_proj: Optional[Union[xr.DataArray, xr.Dataset]] = None,
    diagnostics: Optional[Dict[str, xr.DataArray]] = None,
    eval_quantile: float = 0.50,
    figsize: Optional[Tuple[int, int]] = (16, 28),
) -> Tuple[plt.Figure, np.ndarray]:
    """Plot comprehensive performance diagnostic figures.

    Parameters
    ----------
    ds_obs : xarray.DataArray or xarray.Dataset
        Observed historical baseline dataset.
    ds_ref : xarray.DataArray or xarray.Dataset
        Raw historical model reference dataset.
    ds_qdm_ref : xarray.DataArray or xarray.Dataset
        QDM bias-adjusted historical reference dataset.
    ds_proj : xarray.DataArray or xarray.Dataset, optional
        Raw future model projection dataset.
    ds_qdm_proj : xarray.DataArray or xarray.Dataset, optional
        QDM bias-adjusted future projection dataset.
    diagnostics : dict of {str: xarray.DataArray}, optional
        Pipeline diagnostics dictionary returned by `run_qdm_pipeline`.
    eval_quantile : float, default 0.50
        Quantile level evaluated for heatmaps (Panels 7-12).
    figsize : tuple of int, optional, default (16, 28)
        Dimensions of the output figure in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Created diagnostic figure instance.
    axes : numpy.ndarray
        Array of subplot axes in a 7x2 grid layout.
    """
    if diagnostics:
        q_grid = diagnostics["q_grid"]
    else:
        q_grid = np.linspace(0.001, 0.999, 100)

    units_str = ds_ref.attrs.get("units", "units")
    kind = (
        diagnostics.get("correction_mode", "additive")
        if diagnostics
        else "additive"
    )

    obs_vals = ds_obs.values[~np.isnan(ds_obs.values)]
    ref_vals = ds_ref.values[~np.isnan(ds_ref.values)]
    qdm_ref_vals = ds_qdm_ref.values[~np.isnan(ds_qdm_ref.values)]
    if ds_proj is not None:
        proj_vals = ds_proj.values[~np.isnan(ds_proj.values)]
    else:
        proj_vals = None

    if ds_qdm_proj is not None:
        qdm_proj_vals = ds_qdm_proj.values[~np.isnan(ds_qdm_proj.values)]
    else:
        qdm_proj_vals = None

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(7, 2)

    # 1. Pooled Empirical CDF
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(
        np.quantile(obs_vals, q_grid), q_grid, label="Observed",
        color=COLOR_OBS, lw=5.0, ls="-", alpha=0.5
    )
    ax1.plot(
        np.quantile(ref_vals, q_grid), q_grid, label="Raw Reference",
        color=COLOR_RAW_REF, lw=1.5, ls="--"
    )
    ax1.plot(
        np.quantile(qdm_ref_vals, q_grid), q_grid,
        label="QDM Adjusted Reference", color=COLOR_QDM_REF, lw=1.8, ls="-"
    )
    if proj_vals is not None and qdm_proj_vals is not None:
        ax1.plot(
            np.quantile(proj_vals, q_grid), q_grid, label="Raw Projection",
            color=COLOR_RAW_PROJ, lw=1.5, ls=":"
        )
        ax1.plot(
            np.quantile(qdm_proj_vals, q_grid), q_grid,
            label="QDM Adjusted Projection", color=COLOR_QDM_PROJ,
            lw=2.0, ls="-"
        )
    ax1.set_xlabel(f"Value ({units_str})")
    ax1.set_ylabel("Quantile Probability")
    ax1.set_title("1. Empirical CDF")
    ax1.legend(frameon=True, loc="upper left", fontsize=8.5)
    ax1.grid(True, alpha=0.3)

    # 2. Q-Q Plot
    ax2 = fig.add_subplot(gs[0, 1])
    obs_q = np.quantile(obs_vals, q_grid)
    ref_q = np.quantile(ref_vals, q_grid)
    qdm_ref_q = np.quantile(qdm_ref_vals, q_grid)

    ax2.plot(
        obs_q, ref_q, label="Raw Reference vs Obs",
        color=COLOR_RAW_REF, lw=1.5, ls="--"
    )
    ax2.plot(
        obs_q, qdm_ref_q, label="QDM Adjusted Ref vs Obs",
        color=COLOR_QDM_REF, lw=1.8, ls="-"
    )

    if proj_vals is not None and qdm_proj_vals is not None:
        proj_q = np.quantile(proj_vals, q_grid)
        qdm_proj_q = np.quantile(qdm_proj_vals, q_grid)
        ax2.plot(
            obs_q, proj_q, label="Raw Proj vs Obs",
            color=COLOR_RAW_PROJ, lw=1.5, ls=":"
        )
        ax2.plot(
            obs_q, qdm_proj_q, label="QDM Adjusted Proj vs Obs",
            color=COLOR_QDM_PROJ, lw=2.0, ls="-"
        )

    lim_min = min(obs_q.min(), ref_q.min(), qdm_ref_q.min())
    lim_max = max(obs_q.max(), ref_q.max(), qdm_ref_q.max())
    ax2.plot(
        [lim_min, lim_max], [lim_min, lim_max], color="gray",
        linestyle="-.", alpha=0.7, label="1:1 Line"
    )
    ax2.set_xlabel(f"Observed Quantiles ({units_str})")
    ax2.set_ylabel(f"Model Quantiles ({units_str})")
    ax2.set_title("2. Quantile-Quantile (Q-Q) Plot")
    ax2.legend(frameon=True, loc="upper left", fontsize=8.5)
    ax2.grid(True, alpha=0.3)

    # 3. Reference Period PDF
    ax3 = fig.add_subplot(gs[1, 0])
    v_min_ref = min(obs_vals.min(), ref_vals.min())
    v_max_ref = max(obs_vals.max(), ref_vals.max())
    bins_ref = np.linspace(v_min_ref, v_max_ref, 60)
    
    ax3.hist(
        ref_vals, bins=bins_ref, density=True, alpha=0.25,
        color=COLOR_RAW_REF, edgecolor=COLOR_RAW_REF, linestyle="--",
        label="Raw Reference"
    )
    ax3.hist(
        qdm_ref_vals, bins=bins_ref, density=True, alpha=0.35,
        color=COLOR_QDM_REF, edgecolor=COLOR_QDM_REF, linestyle="-",
        label="QDM Adjusted Reference"
    )
    ax3.hist(
        obs_vals, bins=bins_ref, density=True, histtype="step",
        color=COLOR_OBS, lw=2.0, ls="-", label="Observed Baseline"
    )
    ax3.set_xlabel(f"Value ({units_str})")
    ax3.set_ylabel("Density")
    ax3.set_title("3. Reference Period PDF")
    ax3.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax3.grid(True, alpha=0.3)

    # 4. Projection Period PDF
    ax4 = fig.add_subplot(gs[1, 1])
    if proj_vals is not None and qdm_proj_vals is not None:
        v_min_prj = min(proj_vals.min(), qdm_proj_vals.min())
        v_max_prj = max(proj_vals.max(), qdm_proj_vals.max())
        bins_proj = np.linspace(v_min_prj, v_max_prj, 60)
        
        ax4.hist(
            proj_vals, bins=bins_proj, density=True, alpha=0.25,
            color=COLOR_RAW_PROJ, edgecolor=COLOR_RAW_PROJ, linestyle=":",
            label="Raw Projection"
        )
        ax4.hist(
            qdm_proj_vals, bins=bins_proj, density=True, alpha=0.35,
            color=COLOR_QDM_PROJ, edgecolor=COLOR_QDM_PROJ, linestyle="-",
            label="QDM Adjusted Projection"
        )
    ax4.set_xlabel(f"Value ({units_str})")
    ax4.set_ylabel("Density")
    ax4.set_title("4. Projection Period PDF")
    ax4.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax4.grid(True, alpha=0.3)

    # 5. Day-of-Year Grouped Climatology
    ax5 = fig.add_subplot(gs[2, :])
    def _get_daily_mean(da):
        c = da.sel(time=~((da.time.dt.month == 2) & (da.time.dt.day == 29)))
        return c.groupby("time.dayofyear").mean("time").values[:365]

    ax5.plot(
        np.arange(1, 366), _get_daily_mean(ds_obs), label="Observed",
        color=COLOR_OBS, lw=1.8, ls="-"
    )
    ax5.plot(
        np.arange(1, 366), _get_daily_mean(ds_ref), label="Raw Reference",
        color=COLOR_RAW_REF, lw=1.2, ls="--"
    )
    ax5.plot(
        np.arange(1, 366), _get_daily_mean(ds_qdm_ref),
        label="QDM Adjusted Reference", color=COLOR_QDM_REF, lw=1.5, ls="-"
    )
    if ds_proj is not None and ds_qdm_proj is not None:
        ax5.plot(
            np.arange(1, 366), _get_daily_mean(ds_proj),
            label="Raw Projection", color=COLOR_RAW_PROJ, lw=1.2, ls=":"
        )
        ax5.plot(
            np.arange(1, 366), _get_daily_mean(ds_qdm_proj),
            label="QDM Adjusted Projection", color=COLOR_QDM_PROJ, lw=1.8,
            ls="-"
        )
    ax5.set_xlabel("Day of Year (1..365)")
    ax5.set_ylabel(f"Mean Value ({units_str})")
    ax5.set_title("5. Day-of-Year Grouped Climatology")
    ax5.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax5.grid(True, alpha=0.3)

    # 6. Monthly Mean Diurnal Cycle (PST)
    ax6 = fig.add_subplot(gs[3, :])
    x_c, y_obs, m_c, m_l = _get_monthly_diurnal_cycle_pst(ds_obs)
    _, y_ref, _, _ = _get_monthly_diurnal_cycle_pst(ds_ref)
    _, y_qdm_ref, _, _ = _get_monthly_diurnal_cycle_pst(ds_qdm_ref)

    ax6.plot(
        x_c, y_obs, label="Observed", color=COLOR_OBS, lw=1.8, ls="-"
    )
    ax6.plot(
        x_c, y_ref, label="Raw Reference", color=COLOR_RAW_REF,
        lw=1.2, ls="--"
    )
    ax6.plot(
        x_c, y_qdm_ref, label="QDM Adjusted Reference",
        color=COLOR_QDM_REF, lw=1.5, ls="-"
    )

    if ds_proj is not None and ds_qdm_proj is not None:
        _, y_proj, _, _ = _get_monthly_diurnal_cycle_pst(ds_proj)
        _, y_qdm_proj, _, _ = _get_monthly_diurnal_cycle_pst(ds_qdm_proj)
        ax6.plot(
            x_c, y_proj, label="Raw Projection", color=COLOR_RAW_PROJ,
            lw=1.2, ls=":"
        )
        ax6.plot(
            x_c, y_qdm_proj, label="QDM Adjusted Projection",
            color=COLOR_QDM_PROJ, lw=1.8, ls="-"
        )

    ax6.set_xticks(m_c)
    ax6.set_xticklabels(m_l)
    ax6.set_xlabel("Month (24 Hours / Month, PST)")
    ax6.set_ylabel(f"Mean Value ({units_str})")
    ax6.set_title("6. Monthly Mean Diurnal Cycle (PST)")
    ax6.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax6.grid(True, alpha=0.3)

    # --- July Inset Axis Zoom ---
    jul_center = m_c[6]
    ax_inset = ax6.inset_axes([0.05, 0.45, 0.28, 0.48])
    ax_inset.plot(x_c, y_obs, color=COLOR_OBS, lw=2.0, ls="-")
    ax_inset.plot(x_c, y_ref, color=COLOR_RAW_REF, lw=1.5, ls="--")
    ax_inset.plot(x_c, y_qdm_ref, color=COLOR_QDM_REF, lw=1.8, ls="-")

    if ds_proj is not None and ds_qdm_proj is not None:
        ax_inset.plot(x_c, y_proj, color=COLOR_RAW_PROJ, lw=1.5, ls=":")
        ax_inset.plot(
            x_c, y_qdm_proj, color=COLOR_QDM_PROJ, lw=2.0, ls="-"
        )

    ax_inset.set_xlim(jul_center - 12.0, jul_center + 12.0)
    jul_mask = (x_c >= jul_center - 12.0) & (x_c <= jul_center + 12.0)
    
    y_jul_all = [y_obs[jul_mask], y_ref[jul_mask], y_qdm_ref[jul_mask]]
    if ds_proj is not None and ds_qdm_proj is not None:
        y_jul_all.extend([y_proj[jul_mask], y_qdm_proj[jul_mask]])

    y_jul_stack = np.concatenate([y[~np.isnan(y)] for y in y_jul_all])
    y_margin = (y_jul_stack.max() - y_jul_stack.min()) * 0.12
    ax_inset.set_ylim(
        y_jul_stack.min() - y_margin, y_jul_stack.max() + y_margin
    )

    inset_xt = jul_center - 11.5 + np.array([0, 6, 12, 18, 23])
    ax_inset.set_xticks(inset_xt)
    ax_inset.set_xticklabels(["0", "6", "12", "18", "23"], fontsize=7)
    ax_inset.tick_params(axis="y", labelsize=7)
    ax_inset.set_title("July Zoom (PST)", fontsize=8, pad=3)
    ax_inset.grid(True, alpha=0.3)
    ax6.indicate_inset_zoom(ax_inset, edgecolor="black", alpha=0.45)

# ---------------------------------------------------------------------
    # Heatmap Matrices (Mean Climatologies from Actual 3D Time-Series)
    # ---------------------------------------------------------------------
    q_mat_obs = _roll_to_pst(_compute_hourly_mean_heatmap(ds_obs))
    q_mat_ref = _roll_to_pst(_compute_hourly_mean_heatmap(ds_ref))
    q_mat_qdm_ref = _roll_to_pst(_compute_hourly_mean_heatmap(ds_qdm_ref))

    if ds_proj is not None and ds_qdm_proj is not None:
        q_mat_proj = _roll_to_pst(_compute_hourly_mean_heatmap(ds_proj))
        q_mat_qdm_proj = _roll_to_pst(_compute_hourly_mean_heatmap(ds_qdm_proj))
    else:
        q_mat_proj = q_mat_ref.copy()
        q_mat_qdm_proj = q_mat_qdm_ref.copy()

    # Compute Reference Adjustment Factor (Obs - Ref) and Model Delta (Proj - Ref)
    if kind == "additive":
        q_ref_factor = q_mat_obs - q_mat_ref
        q_raw_delta = q_mat_proj - q_mat_ref
        cmap_adj_factor = "PuOr_r"
        label_adj_factor = f"Adjustment Factor ({units_str})"
        cmap_delta = "PuOr_r"
        label_delta = f"Model Delta ({units_str})"
    else:
        q_ref_factor = q_mat_obs / np.where(q_mat_ref == 0, 1e-6, q_mat_ref)
        q_raw_delta = q_mat_proj / np.where(q_mat_ref == 0, 1e-6, q_mat_ref)
        cmap_adj_factor = "PiYG"
        label_adj_factor = "Adjustment Ratio (Multiplier)"
        cmap_delta = "PiYG"
        label_delta = "Model Delta Ratio (Multiplier)"

    vmax_q = max(
        q_mat_ref.max(), q_mat_qdm_ref.max(),
        q_mat_proj.max(), q_mat_qdm_proj.max()
    )
    vmin_q = min(
        q_mat_ref.min(), q_mat_qdm_ref.min(),
        q_mat_proj.min(), q_mat_qdm_proj.min()
    )

    # 7. Raw Reference Heatmap
    ax7 = fig.add_subplot(gs[4, 0])
    im7 = ax7.imshow(
        q_mat_ref, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im7, ax=ax7, label=f"Mean ({units_str})", shrink=0.85)
    ax7.set_title("7. Raw Reference Mean Climatology")

    # 8. QDM Adjusted Reference Heatmap
    ax8 = fig.add_subplot(gs[4, 1])
    im8 = ax8.imshow(
        q_mat_qdm_ref, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im8, ax=ax8, label=f"Mean ({units_str})", shrink=0.85)
    ax8.set_title("8. QDM Adj Reference Mean Climatology")

    # 9. Raw Projection Heatmap
    ax9 = fig.add_subplot(gs[5, 0])
    im9 = ax9.imshow(
        q_mat_proj, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im9, ax=ax9, label=f"Mean ({units_str})", shrink=0.85)
    ax9.set_title("9. Raw Projection Mean Climatology")

    # 10. QDM Adjusted Projection Heatmap
    ax10 = fig.add_subplot(gs[5, 1])
    im10 = ax10.imshow(
        q_mat_qdm_proj, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im10, ax=ax10, label=f"Mean ({units_str})", shrink=0.85)
    ax10.set_title("10. QDM Adj Projection Mean Climatology")

    # 11. Reference Adjustment Factor Heatmap
    ax11 = fig.add_subplot(gs[6, 0])
    if kind == "additive":
        vmax_adj = max(0.01, np.max(np.abs(q_ref_factor)))
        vmin_adj = -vmax_adj
    else:
        vmax_adj = max(1.01, np.max(q_ref_factor))
        vmin_adj = min(0.99, np.min(q_ref_factor))

    im11 = ax11.imshow(
        q_ref_factor, aspect="auto", origin="lower", cmap=cmap_adj_factor,
        vmin=vmin_adj, vmax=vmax_adj, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im11, ax=ax11, label=label_adj_factor, shrink=0.85)
    ax11.set_title("11. Ref Mean Adjustment Factor (Obs - Ref)")

    # 12. Model Delta Heatmap
    ax12 = fig.add_subplot(gs[6, 1])
    if kind == "additive":
        vmax_delta = max(0.01, np.max(np.abs(q_raw_delta)))
        vmin_delta = -vmax_delta
    else:
        vmax_delta = max(1.01, np.max(q_raw_delta))
        vmin_delta = min(0.99, np.min(q_raw_delta))

    im12 = ax12.imshow(
        q_raw_delta, aspect="auto", origin="lower", cmap=cmap_delta,
        vmin=vmin_delta, vmax=vmax_delta, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im12, ax=ax12, label=label_delta, shrink=0.85)
    ax12.set_title("12. Model Mean Delta (Proj - Ref)")

    for ax in [ax7, ax8, ax9, ax10, ax11, ax12]:
        ax.set_xlabel("Day of Year")
        ax.set_ylabel("Hour of Day (PST)")

    axes = np.array([
        [ax1, ax2], [ax3, ax4], [ax5, ax5], [ax6, ax6],
        [ax7, ax8], [ax9, ax10], [ax11, ax12]
    ])
    return fig, axes


def plot_qdm_pooled_heatmaps(
    diagnostics: Dict[str, xr.DataArray],
    eval_quantile: float = 0.50,
    units_str: str = "degF",
    figsize: Tuple[int, int] = (16, 16),
) -> Tuple[plt.Figure, np.ndarray]:
    """Plot standalone heatmaps displaying internal QDM pooled quantile matrices.

    Parameters
    ----------
    diagnostics : dict of {str: xarray.DataArray}
        Pipeline diagnostics dictionary returned by `run_qdm_pipeline`.
    eval_quantile : float, default 0.50
        Quantile probability level (0.0 to 1.0) evaluated across heatmaps.
    units_str : str, default 'degF'
        Units label for plot colorbars.
    figsize : tuple of int, default (16, 16)
        Dimensions (width, height) of the output figure in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Created standalone diagnostic figure instance.
    axes : numpy.ndarray
        Array of subplot axes in a 4x2 grid layout.
    """
    kind = diagnostics.get("correction_mode", "additive")

    # 1. Extract pooled quantile matrices directly from pipeline diagnostics
    q_mat_obs = _roll_to_pst(
        _extract_quantile_heatmap(diagnostics, "q_obs", eval_quantile)
    )
    q_mat_ref = _roll_to_pst(
        _extract_quantile_heatmap(diagnostics, "q_ref", eval_quantile)
    )

    if "q_proj" in diagnostics:
        q_mat_proj = _roll_to_pst(
            _extract_quantile_heatmap(diagnostics, "q_proj", eval_quantile)
        )
    else:
        q_mat_proj = q_mat_ref.copy()

    # In quantile space, QDM maps the reference model directly onto the observed baseline
    q_mat_qdm_ref = q_mat_obs.copy()

    # Derive QDM adjusted projection matrix in quantile space
    if kind == "additive":
        q_mat_qdm_proj = q_mat_obs + (q_mat_proj - q_mat_ref)
        q_ref_factor = q_mat_obs - q_mat_ref
        q_raw_delta = q_mat_proj - q_mat_ref
        cmap_adj_factor = "PuOr_r"
        label_adj_factor = f"Adjustment Factor ({units_str})"
        cmap_delta = "PuOr_r"
        label_delta = f"Model Delta ({units_str})"
    else:
        q_mat_qdm_proj = q_mat_obs * (
            q_mat_proj / np.where(q_mat_ref == 0, 1e-6, q_mat_ref)
        )
        q_ref_factor = q_mat_obs / np.where(
            q_mat_ref == 0, 1e-6, q_mat_ref
        )
        q_raw_delta = q_mat_proj / np.where(
            q_mat_ref == 0, 1e-6, q_mat_ref
        )
        cmap_adj_factor = "PiYG"
        label_adj_factor = "Adjustment Ratio (Multiplier)"
        cmap_delta = "PiYG"
        label_delta = "Model Delta Ratio (Multiplier)"

    vmax_q = max(
        q_mat_ref.max(), q_mat_qdm_ref.max(),
        q_mat_proj.max(), q_mat_qdm_proj.max()
    )
    vmin_q = min(
        q_mat_ref.min(), q_mat_qdm_ref.min(),
        q_mat_proj.min(), q_mat_qdm_proj.min()
    )

    # 2. Construct 4x2 subplot layout
    fig, axes = plt.subplots(4, 2, figsize=figsize, constrained_layout=True)

    # Panel 1: Observed Pooled Quantiles
    im0 = axes[0, 0].imshow(
        q_mat_obs, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im0, ax=axes[0, 0], label=f"Value ({units_str})", shrink=0.85)
    axes[0, 0].set_title(f"1. Obs Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 2: Adjustment Factor Matrix
    if kind == "additive":
        vmax_adj = max(0.01, np.max(np.abs(q_ref_factor)))
        vmin_adj = -vmax_adj
    else:
        vmax_adj = max(1.01, np.max(q_ref_factor))
        vmin_adj = min(0.99, np.min(q_ref_factor))

    im1 = axes[0, 1].imshow(
        q_ref_factor, aspect="auto", origin="lower", cmap=cmap_adj_factor,
        vmin=vmin_adj, vmax=vmax_adj, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im1, ax=axes[0, 1], label=label_adj_factor, shrink=0.85)
    axes[0, 1].set_title(f"2. QDM Adj Factor Matrix (q = {eval_quantile:.2f})")

    # Panel 3: Raw Reference Pooled Quantiles
    im2 = axes[1, 0].imshow(
        q_mat_ref, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im2, ax=axes[1, 0], label=f"Value ({units_str})", shrink=0.85)
    axes[1, 0].set_title(f"3. Raw Ref Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 4: QDM Adjusted Reference Pooled Quantiles
    im3 = axes[1, 1].imshow(
        q_mat_qdm_ref, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im3, ax=axes[1, 1], label=f"Value ({units_str})", shrink=0.85)
    axes[1, 1].set_title(f"4. QDM Adj Ref Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 5: Raw Projection Pooled Quantiles
    im4 = axes[2, 0].imshow(
        q_mat_proj, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im4, ax=axes[2, 0], label=f"Value ({units_str})", shrink=0.85)
    axes[2, 0].set_title(f"5. Raw Proj Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 6: QDM Adjusted Projection Pooled Quantiles
    im5 = axes[2, 1].imshow(
        q_mat_qdm_proj, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im5, ax=axes[2, 1], label=f"Value ({units_str})", shrink=0.85)
    axes[2, 1].set_title(f"6. QDM Adj Proj Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 7: Model Delta Matrix
    if kind == "additive":
        vmax_delta = max(0.01, np.max(np.abs(q_raw_delta)))
        vmin_delta = -vmax_delta
    else:
        vmax_delta = max(1.01, np.max(q_raw_delta))
        vmin_delta = min(0.99, np.min(q_raw_delta))

    im6 = axes[3, 0].imshow(
        q_raw_delta, aspect="auto", origin="lower", cmap=cmap_delta,
        vmin=vmin_delta, vmax=vmax_delta, extent=[1, 365, 0, 23]
    )
    fig.colorbar(im6, ax=axes[3, 0], label=label_delta, shrink=0.85)
    axes[3, 0].set_title(f"7. Model Delta Matrix (q = {eval_quantile:.2f})")

    # Panel 8: Hide empty unused cell in 4x2 grid
    axes[3, 1].axis("off")

    for ax in axes.flat[:-1]:
        ax.set_xlabel("Day of Year")
        ax.set_ylabel("Hour of Day (PST)")

    return fig, axes

# Color Palette for Sensitivity / Delta Analysis
COLOR_OBS_RAW = "#000000"  # Solid Black (Raw Observed Baseline)
COLOR_OBS_SMOOTH = "#666666"  # Medium Gray / Dashed (Smoothed Observed Baseline)
COLOR_RAW = "#999999"  # Light Gray (Raw Model Baseline)
COLOR_PIPE_A = "#D55E00"  # Vermillion/Orange (Pipeline A: Smoothed Obs)
COLOR_PIPE_B = "#0072B2"  # Deep Blue (Pipeline B: Raw/Unsmoothed Obs)


def plot_qdm_sensitivity_deltas(
    ds_obs_raw: xr.DataArray,
    ds_obs_smooth: xr.DataArray,
    ds_ref_raw: xr.DataArray,
    ds_qdm_ref_a: xr.DataArray,
    ds_qdm_ref_b: xr.DataArray,
    diag_a: Dict[str, xr.DataArray],
    diag_b: Dict[str, xr.DataArray],
    label_a: str = "Smoothed Obs QDM",
    label_b: str = "Raw Obs QDM",
    ds_proj_raw: Optional[xr.DataArray] = None,
    ds_qdm_proj_a: Optional[xr.DataArray] = None,
    ds_qdm_proj_b: Optional[xr.DataArray] = None,
    eval_quantile: float = 0.50,
    pst_offset: int = -8,
    figsize: Optional[Tuple[int, int]] = (16, 26),
) -> Tuple[plt.Figure, np.ndarray]:
    """Plot sensitivity diagnostic suite comparing QDM pipelines run on smoothed

    vs. raw/unsmoothed observations.
    """
        # 1. Input Data Check
    print(
        "Input Obs Diff Max:", float(np.abs(ds_obs_raw - ds_obs_smooth).max().values)
    )
    
    # 2. Diagnostic Object Check
    q_obs_a = diag_a["q_obs"]
    q_obs_b = diag_b["q_obs"]
    print(
        "Quantile Obs Diff Max:",
        float(np.abs(q_obs_b.values - q_obs_a.values).max()),
    )
    
    # 3. Heatmap Matrix Extraction Check
    m_a = _extract_quantile_heatmap(diag_a, "q_obs", eval_quantile=eval_quantile)
    m_b = _extract_quantile_heatmap(diag_b, "q_obs", eval_quantile=eval_quantile)
    print("Extracted Heatmap A Max:", np.max(np.abs(m_a)))
    print("Extracted Heatmap B Max:", np.max(np.abs(m_b)))
    print("Extracted Heatmap Delta Max:", np.max(np.abs(m_b - m_a)))
    
    units_str = ds_ref_raw.attrs.get("units", "units")
    kind = diag_a.get("correction_mode", "additive") if diag_a else "additive"

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = gridspec.GridSpec(6, 2, figure=fig)

    # -------------------------------------------------------------------------
    # Row 1: PDF Density Residuals & Baseline Overlay (Panels 1 & 2)
    # -------------------------------------------------------------------------
    ref_a_vals = ds_qdm_ref_a.values[~np.isnan(ds_qdm_ref_a.values)]
    ref_b_vals = ds_qdm_ref_b.values[~np.isnan(ds_qdm_ref_b.values)]
    obs_raw_vals = ds_obs_raw.values[~np.isnan(ds_obs_raw.values)]
    obs_smooth_vals = ds_obs_smooth.values[~np.isnan(ds_obs_smooth.values)]

    ax1 = fig.add_subplot(gs[0, 0])

    # PDF Residual Delta between Pipelines
    bins_ref = np.linspace(
        min(ref_a_vals.min(), ref_b_vals.min()),
        max(ref_a_vals.max(), ref_b_vals.max()),
        50,
    )
    counts_a_ref, _ = np.histogram(ref_a_vals, bins=bins_ref, density=True)
    counts_b_ref, _ = np.histogram(ref_b_vals, bins=bins_ref, density=True)
    delta_pdf_ref = counts_b_ref - counts_a_ref

    bin_centers_ref = (bins_ref[:-1] + bins_ref[1:]) / 2.0
    bin_width_ref = bins_ref[1] - bins_ref[0]
    colors_bar_ref = np.where(delta_pdf_ref >= 0, COLOR_PIPE_B, COLOR_PIPE_A)

    ax1.bar(
        bin_centers_ref,
        delta_pdf_ref,
        width=bin_width_ref * 0.9,
        color=colors_bar_ref,
        alpha=0.75,
        edgecolor="black",
        linewidth=0.5,
    )

    # Overlay Observation PDF comparison on twin axis
    ax1_twin = ax1.twinx()
    counts_obs_raw, _ = np.histogram(
        obs_raw_vals, bins=bins_ref, density=True
    )
    counts_obs_smooth, _ = np.histogram(
        obs_smooth_vals, bins=bins_ref, density=True
    )

    ax1_twin.plot(
        bin_centers_ref,
        counts_obs_raw,
        color=COLOR_OBS_RAW,
        lw=1.5,
        label="Raw Obs PDF",
    )
    ax1_twin.plot(
        bin_centers_ref,
        counts_obs_smooth,
        color=COLOR_OBS_SMOOTH,
        lw=1.5,
        ls="--",
        label="Smooth Obs PDF",
    )
    ax1_twin.set_ylabel("Obs Density", color=COLOR_OBS_RAW)
    ax1_twin.tick_params(axis="y", labelcolor=COLOR_OBS_RAW)

    ax1.axhline(0, color="black", linestyle="--", alpha=0.7, lw=1.0)
    ax1.set_xlabel(f"Value ({units_str})")
    ax1.set_ylabel(r"$\Delta$ Density Residual")
    ax1.set_title(f"1. Ref PDF Density Delta ({label_b} - {label_a})")

    legend_patches_ref = [
        mpatches.Patch(
            facecolor=COLOR_PIPE_B,
            alpha=0.75,
            edgecolor="black",
            label=f"Higher in {label_b}",
        ),
        mpatches.Patch(
            facecolor=COLOR_PIPE_A,
            alpha=0.75,
            edgecolor="black",
            label=f"Higher in {label_a}",
        ),
    ]
    ax1.legend(
        handles=legend_patches_ref,
        frameon=True,
        loc="upper right",
        fontsize=8.5,
    )
    ax1.grid(True, alpha=0.3)

    # Panel 2: Projection PDF Delta
    ax2 = fig.add_subplot(gs[0, 1])
    delta_pdf_proj = np.array([0.0])

    if ds_qdm_proj_a is not None and ds_qdm_proj_b is not None:
        proj_a_vals = ds_qdm_proj_a.values[~np.isnan(ds_qdm_proj_a.values)]
        proj_b_vals = ds_qdm_proj_b.values[~np.isnan(ds_qdm_proj_b.values)]

        bins_proj = np.linspace(
            min(proj_a_vals.min(), proj_b_vals.min()),
            max(proj_a_vals.max(), proj_b_vals.max()),
            50,
        )
        counts_a_proj, _ = np.histogram(
            proj_a_vals, bins=bins_proj, density=True
        )
        counts_b_proj, _ = np.histogram(
            proj_b_vals, bins=bins_proj, density=True
        )
        delta_pdf_proj = counts_b_proj - counts_a_proj

        bin_centers_proj = (bins_proj[:-1] + bins_proj[1:]) / 2.0
        bin_width_proj = bins_proj[1] - bins_proj[0]
        colors_bar_proj = np.where(
            delta_pdf_proj >= 0, COLOR_PIPE_B, COLOR_PIPE_A
        )

        ax2.bar(
            bin_centers_proj,
            delta_pdf_proj,
            width=bin_width_proj * 0.9,
            color=colors_bar_proj,
            alpha=0.75,
            edgecolor="black",
            linewidth=0.5,
        )
        ax2.axhline(0, color="black", linestyle="--", alpha=0.7, lw=1.0)
        ax2.set_xlabel(f"Value ({units_str})")
        ax2.set_ylabel(r"$\Delta$ Density Residual")
        ax2.set_title(f"2. Proj PDF Density Delta ({label_b} - {label_a})")

        legend_patches_proj = [
            mpatches.Patch(
                facecolor=COLOR_PIPE_B,
                alpha=0.75,
                edgecolor="black",
                label=f"Higher in {label_b}",
            ),
            mpatches.Patch(
                facecolor=COLOR_PIPE_A,
                alpha=0.75,
                edgecolor="black",
                label=f"Higher in {label_a}",
            ),
        ]
        ax2.legend(
            handles=legend_patches_proj,
            frameon=True,
            loc="upper right",
            fontsize=8.5,
        )

    ax2.grid(True, alpha=0.3)

    # Harmonize Y-axis limits on PDF panels
    y_max_pdf = max(np.max(delta_pdf_ref), np.max(delta_pdf_proj))
    y_min_pdf = min(np.min(delta_pdf_ref), np.min(delta_pdf_proj))
    pdf_range_pad = (
        (y_max_pdf - y_min_pdf) * 0.10 if y_max_pdf != y_min_pdf else 0.01
    )
    ax1.set_ylim(y_min_pdf - pdf_range_pad, y_max_pdf + pdf_range_pad)
    ax2.set_ylim(y_min_pdf - pdf_range_pad, y_max_pdf + pdf_range_pad)

    # -------------------------------------------------------------------------
    # Row 2: Day-of-Year Mean Residual Delta (Panel 3)
    # -------------------------------------------------------------------------
    ax3 = fig.add_subplot(gs[1, :])

    def _get_daily_mean(da):
        clean = da.sel(
            time=~((da.time.dt.month == 2) & (da.time.dt.day == 29))
        )
        return clean.groupby("time.dayofyear").mean("time").values[:365]

    mean_a_ref = _get_daily_mean(ds_qdm_ref_a)
    mean_b_ref = _get_daily_mean(ds_qdm_ref_b)
    mean_obs_raw = _get_daily_mean(ds_obs_raw)
    mean_obs_smooth = _get_daily_mean(ds_obs_smooth)

    # Plot pipeline differences
    ax3.plot(
        np.arange(1, 366),
        mean_b_ref - mean_a_ref,
        color=COLOR_PIPE_A,
        lw=2.2,
        label=f"Ref Delta ({label_b} - {label_a})",
    )

    if ds_qdm_proj_a is not None and ds_qdm_proj_b is not None:
        mean_a_proj = _get_daily_mean(ds_qdm_proj_a)
        mean_b_proj = _get_daily_mean(ds_qdm_proj_b)
        ax3.plot(
            np.arange(1, 366),
            mean_b_proj - mean_a_proj,
            color=COLOR_PIPE_B,
            lw=2.2,
            ls="--",
            label=f"Proj Delta ({label_b} - {label_a})",
        )

    # Plot observational baseline delta (Raw Obs - Smooth Obs)
    ax3.plot(
        np.arange(1, 366),
        mean_obs_raw - mean_obs_smooth,
        color=COLOR_OBS_RAW,
        lw=1.5,
        ls=":",
        label="Obs Baseline Delta (Raw - Smooth)",
    )

    ax3.axhline(0, color="black", linestyle=":", alpha=0.6)
    ax3.set_xlabel("Day of Year (1..365)")
    ax3.set_ylabel(f"$\Delta$ Mean Value ({units_str})")
    ax3.set_title(
        f"3. Day-of-Year Mean Residual Delta ({label_b} - {label_a})"
    )
    ax3.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax3.grid(True, alpha=0.3)

    # -------------------------------------------------------------------------
    # Row 3: Month-of-Year Diurnal Cycle Residual Delta (Panel 4)
    # -------------------------------------------------------------------------
    ax4 = fig.add_subplot(gs[2, :])

    x_coords, diurnal_a_ref, month_centers, month_labels = (
        _get_monthly_diurnal_cycle_pst(ds_qdm_ref_a, pst_offset=pst_offset)
    )
    _, diurnal_b_ref, _, _ = _get_monthly_diurnal_cycle_pst(
        ds_qdm_ref_b, pst_offset=pst_offset
    )

    ax4.plot(
        x_coords,
        diurnal_b_ref - diurnal_a_ref,
        color=COLOR_PIPE_A,
        lw=2.0,
        label=f"Ref Diurnal Delta ({label_b} - {label_a})",
    )

    if ds_qdm_proj_a is not None and ds_qdm_proj_b is not None:
        _, diurnal_a_proj, _, _ = _get_monthly_diurnal_cycle_pst(
            ds_qdm_proj_a, pst_offset=pst_offset
        )
        _, diurnal_b_proj, _, _ = _get_monthly_diurnal_cycle_pst(
            ds_qdm_proj_b, pst_offset=pst_offset
        )
        ax4.plot(
            x_coords,
            diurnal_b_proj - diurnal_a_proj,
            color=COLOR_PIPE_B,
            lw=2.0,
            ls="--",
            label=f"Proj Diurnal Delta ({label_b} - {label_a})",
        )

    ax4.set_xticks(month_centers)
    ax4.set_xticklabels(month_labels)
    ax4.axhline(0, color="black", linestyle=":", alpha=0.6)
    ax4.set_xlabel("Month (24 Hours / Month, PST)")
    ax4.set_ylabel(f"$\Delta$ Mean Value ({units_str})")
    ax4.set_title(
        f"4. Month-of-Year Diurnal Cycle Mean Residual Delta "
        f"({label_b} - {label_a})"
    )
    ax4.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax4.grid(True, alpha=0.3)

    # -------------------------------------------------------------------------
    # Heatmap Matrix Extractions & Rolled PST Operations
    # -------------------------------------------------------------------------
    q_mat_a_ref = _roll_to_pst(
        _extract_quantile_heatmap(diag_a, "q_ref", eval_quantile), pst_offset
    )
    q_mat_b_ref = _roll_to_pst(
        _extract_quantile_heatmap(diag_b, "q_ref", eval_quantile), pst_offset
    )
    q_mat_a_proj = _roll_to_pst(
        _extract_quantile_heatmap(diag_a, "q_proj", eval_quantile), pst_offset
    )
    q_mat_b_proj = _roll_to_pst(
        _extract_quantile_heatmap(diag_b, "q_proj", eval_quantile), pst_offset
    )

    # Extract distinct observed quantiles from diagnostics dictionaries
    q_mat_obs_smooth = _roll_to_pst(
        _extract_quantile_heatmap(diag_a, "q_obs", eval_quantile), pst_offset
    )
    q_mat_obs_raw = _roll_to_pst(
        _extract_quantile_heatmap(diag_b, "q_obs", eval_quantile), pst_offset
    )

    delta_ref_mat = q_mat_b_ref - q_mat_a_ref
    delta_proj_mat = q_mat_b_proj - q_mat_a_proj

    proj_ref_delta_a = q_mat_a_proj - q_mat_a_ref
    proj_ref_delta_b = q_mat_b_proj - q_mat_b_ref

    # Adjustment Factors (Obs - Ref) evaluating smoothed vs raw obs
    if kind == "additive":
        adj_factor_a = q_mat_obs_smooth - q_mat_a_ref
        adj_factor_b = q_mat_obs_raw - q_mat_b_ref
        label_factor_delta = f"$\Delta$ Adjustment Factor ({units_str})"
    else:
        adj_factor_a = q_mat_obs_smooth / np.where(
            q_mat_a_ref == 0, 1e-6, q_mat_a_ref
        )
        adj_factor_b = q_mat_obs_raw / np.where(
            q_mat_b_ref == 0, 1e-6, q_mat_b_ref
        )
        label_factor_delta = r"$\Delta$ Adjustment Ratio"

    delta_adj_factor = adj_factor_b - adj_factor_a

    # Scale bounds
    vmax_delta_ref_proj = max(
        0.01,
        np.max(np.abs(delta_ref_mat)),
        np.max(np.abs(delta_proj_mat)),
    )
    vmax_proj_ref_delta = max(
        0.01,
        np.max(np.abs(proj_ref_delta_a)),
        np.max(np.abs(proj_ref_delta_b)),
    )
    vmax_adj_factor_delta = max(0.01, np.max(np.abs(delta_adj_factor)))

    # -------------------------------------------------------------------------
    # Row 4: Delta Ref & Proj Heatmaps (Panels 5 & 6)
    # -------------------------------------------------------------------------
    ax5 = fig.add_subplot(gs[3, 0])
    im5 = ax5.imshow(
        delta_ref_mat,
        aspect="auto",
        origin="lower",
        cmap="PuOr_r",
        vmin=-vmax_delta_ref_proj,
        vmax=vmax_delta_ref_proj,
        extent=[1, 365, 0, 23],
    )
    fig.colorbar(
        im5, ax=ax5, label=f"$\Delta$ Value ({units_str})", shrink=0.85
    )
    ax5.set_title(
        f"5. Delta Ref Heatmap ({label_b} - {label_a}, q = {eval_quantile:.2f})"
    )

    ax6 = fig.add_subplot(gs[3, 1])
    im6 = ax6.imshow(
        delta_proj_mat,
        aspect="auto",
        origin="lower",
        cmap="PuOr_r",
        vmin=-vmax_delta_ref_proj,
        vmax=vmax_delta_ref_proj,
        extent=[1, 365, 0, 23],
    )
    fig.colorbar(
        im6, ax=ax6, label=f"$\Delta$ Value ({units_str})", shrink=0.85
    )
    ax6.set_title(
        f"6. Delta Proj Heatmap ({label_b} - {label_a}, q = {eval_quantile:.2f})"
    )

    # -------------------------------------------------------------------------
    # Row 5: Delta Proj-Ref for Pipeline A and B (Panels 7 & 8)
    # -------------------------------------------------------------------------
    ax7 = fig.add_subplot(gs[4, 0])
    im7 = ax7.imshow(
        proj_ref_delta_a,
        aspect="auto",
        origin="lower",
        cmap="PuOr_r",
        vmin=-vmax_proj_ref_delta,
        vmax=vmax_proj_ref_delta,
        extent=[1, 365, 0, 23],
    )
    fig.colorbar(im7, ax=ax7, label=f"Model Delta ({units_str})", shrink=0.85)
    ax7.set_title(f"7. Delta Proj-Ref: {label_a} (q = {eval_quantile:.2f})")

    ax8 = fig.add_subplot(gs[4, 1])
    im8 = ax8.imshow(
        proj_ref_delta_b,
        aspect="auto",
        origin="lower",
        cmap="PuOr_r",
        vmin=-vmax_proj_ref_delta,
        vmax=vmax_proj_ref_delta,
        extent=[1, 365, 0, 23],
    )
    fig.colorbar(im8, ax=ax8, label=f"Model Delta ({units_str})", shrink=0.85)
    ax8.set_title(f"8. Delta Proj-Ref: {label_b} (q = {eval_quantile:.2f})")

    # -------------------------------------------------------------------------
    # Row 6: Standalone Delta Adjustment Factor Heatmap (Panel 9)
    # -------------------------------------------------------------------------
    ax9 = fig.add_subplot(gs[5, 0])
    im9 = ax9.imshow(
        delta_adj_factor,
        aspect="auto",
        origin="lower",
        cmap="PuOr_r",
        vmin=-vmax_adj_factor_delta,
        vmax=vmax_adj_factor_delta,
        extent=[1, 365, 0, 23],
    )
    fig.colorbar(im9, ax=ax9, label=label_factor_delta, shrink=0.85)
    ax9.set_title(
        f"9. Delta Adjustment Factor ({label_b} - {label_a}, "
        f"q = {eval_quantile:.2f})"
    )

    for ax in [ax5, ax6, ax7, ax8, ax9]:
        ax.set_xlabel("Day of Year")
        ax.set_ylabel("Hour of Day (PST)")

    axes = np.array(
        [
            [ax1, ax2],
            [ax3, ax3],
            [ax4, ax4],
            [ax5, ax6],
            [ax7, ax8],
            [ax9, None],
        ]
    )
    return fig, axes

In [ ]:
# Set 30-year reference analysis temporal bounds
ref_slice = ["1992-01-01T00:00:00", "2021-12-31T23:00:00"]

all_stns = gpd.read_file("hdp_wrf_locs_d03.gdb/")
stn_name = "KSAN"
stn_gdf = all_stns[all_stns["source-id"] == stn_name]

my_station = stn_gdf["era-id"]
my_network = stn_gdf["network"]

# Load in the raw observations
# Query Intake datastore
cat = intake.open_esm_datastore(
    "https://cadcat.s3.amazonaws.com/histwxstns/era-hdp-collection.json"
)
query = {"station_id": my_station.item()}
cat_subset = cat.search(**query)

dsets = cat_subset.to_dataset_dict(
    xarray_open_kwargs={"consolidated": False}, storage_options={"anon": True}
)
obs_ds = list(dsets.values())[0]

# Subset slice and target variables
obs_ds = obs_ds.sel(time=slice(ref_slice[0], ref_slice[1]))
obs_ds = obs_ds[
    [
        "tdps",
        "tdps_eraqc",
    ]
]

obs_ds = obs_ds.convert_calendar("noleap")
obs_ds['tdps'] = convert_units_to(obs_ds['tdps'], 'degF')
obs_ds = obs_ds.squeeze()

# Remove observations with QAQC flag
obs_ds = apply_qaqc(obs_ds, stn_name=stn_name)
obs_ds = obs_ds.tdps.compute()

# Load in the smoothed observations
obs_ds_smooth = xr.open_dataset('ksan_smoothed_tdps/',engine="zarr")
obs_ds_smooth = obs_ds_smooth.tdps.compute()


In [ ]:
cd = ck.ClimateData(verbosity=-2)
time_slice = (1981, 2075)
wrf_ds = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .variable("dew_point")
    .processes({
        "clip": "KSAN",
        "convert_units": "degF",
        "time_slice": time_slice
    })
    .get()
)
display(wrf_ds)

In [ ]:
# wrf_ds = wrf_ds.sel(sim = 'wrf_ucla_ec-earth3-veg_ssp370_r1i1p1f1').squeeze()
wrf_ds = wrf_ds.sel(sim = 'wrf_ucla_taiesm1_ssp370_r1i1p1f1').squeeze()

wrf_ds = wrf_ds["dew_point"]
drop_list = list(c for c in wrf_ds.coords if c!="time")
wrf_ds = wrf_ds.convert_calendar("noleap")
wrf_ds = wrf_ds.drop_vars(drop_list)

hist_ds = wrf_ds.sel(
        time=slice(str(obs_ds.time.values[0]),
        str(obs_ds.time.values[-1]))).compute()
sim_ds = wrf_ds.sel(
    time=slice("2046-01-01 00:00:00",
               "2075-12-31 23:00:00")).compute()

# Raw vs. smoothed observations QDM sensitivity tests

## Day-of-year QDM

In [ ]:
%%time

# Run QDM
# Start with raw (unsmoothed) observations
ds_qdm_proj_raw, ds_qdm_ref_raw, diagnostics_raw = run_qdm_pipeline(
    ds_obs=obs_ds,
    ds_ref=hist_ds,
    ds_proj=sim_ds,
    hour_window=3,
    day_window=61,
    return_intermediate=True,
)

### Under-the-hood: The statistics which the QDM "sees"



In [ ]:
# Under-the-hood pooled heatmaps
fig_under_hood_raw, axes_under_hood_raw = plot_qdm_pooled_heatmaps(
    diagnostics_raw, eval_quantile=0.95, units_str="degF"
)

### Results

In [ ]:
# Plot diagnostic suite
fig, axes = plot_qdm_diagnostics(
    ds_obs=obs_ds,
    ds_ref=hist_ds,
    ds_qdm_ref=ds_qdm_ref_raw,
    ds_proj=sim_ds,
    ds_qdm_proj=ds_qdm_proj_raw,
    diagnostics=diagnostics_raw,
    eval_quantile=0.95,
)

In [ ]:
%%time

# Run QDM
# QDM with smoothed observations
ds_qdm_proj_smooth, ds_qdm_ref_smooth, diagnostics_smooth = run_qdm_pipeline(
    ds_obs=obs_ds_smooth,
    ds_ref=hist_ds,
    ds_proj=sim_ds,
    hour_window=3,
    day_window=61,
    return_intermediate=True,
)


### Under-the-hood: The statistics which the QDM "sees"

In [ ]:
# Under-the-hood pooled heatmaps
fig_under_hood_smooth, axes_under_hood_smooth = plot_qdm_pooled_heatmaps(
    diagnostics_smooth, eval_quantile=0.95, units_str="degF"
)

In [ ]:
# Plot diagnostic suite
fig, axes = plot_qdm_diagnostics(
    ds_obs=obs_ds_smooth,
    ds_ref=hist_ds,
    ds_qdm_ref=ds_qdm_ref_smooth,
    ds_proj=sim_ds,
    ds_qdm_proj=ds_qdm_proj_smooth,
    diagnostics=diagnostics_smooth,
    eval_quantile=0.95,
)

## Intercomparison diagnostics

In [ ]:
fig, axes = plot_qdm_sensitivity_deltas(
    ds_obs_raw=obs_ds,
    ds_obs_smooth=obs_ds_smooth,
    ds_ref_raw=hist_ds,
    ds_qdm_ref_a=ds_qdm_ref_raw,
    ds_qdm_ref_b=ds_qdm_ref_smooth,
    diag_a=diagnostics_raw,
    diag_b=diagnostics_smooth,
    label_a="Raw QDM",
    label_b="Smoothed QDM",
    ds_proj_raw=sim_ds,
    ds_qdm_proj_a=ds_qdm_proj_raw,
    ds_qdm_proj_b=ds_qdm_proj_smooth,
    eval_quantile=0.99,
)

fig.suptitle(
    "QDM Sensitivity Test: Raw  vs. Smoothed Observations",
    fontsize=16,
    y=1.01,
)